**Importer les libraries pertinentes**

In [1]:
import pandas as pd
from os import listdir
from os import path

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Naive Bayes
from sklearn.naive_bayes import MultinomialNB

# K plus proches voisins (kNN)
from sklearn.neighbors import KNeighborsClassifier

**Entraîner les modèles et sortir les résultats**

In [2]:
folder = '../1-data/training_datasets/'
datasets = listdir(folder)
datasets

['train_dataset_10pc.xlsx',
 'train_dataset_20pc.xlsx',
 'train_dataset_30pc.xlsx',
 'train_dataset_40pc.xlsx',
 'train_dataset_50pc.xlsx',
 'train_dataset_60pc.xlsx',
 'train_dataset_70pc.xlsx',
 'train_dataset_80pc.xlsx',
 'train_dataset_90pc.xlsx']

In [3]:
report = []
for file in datasets:
    ratio = file[14:-7]
    
    df = pd.read_excel(path.join(folder, file))
    X = df['text_post'].astype(str)
    y = df['category']


    # 5-fold cross-validation
    stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Valeurs du paramètre k à tester pour les classifieurs knn
    k_values = [1, 2, 3, 4, 5]

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [100, 250, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000]

    ngram_values = (1, 2)

    # Antidictionnaire personnalisé
    with open('./utils/exclusion_v2.stop', encoding='utf-8') as f:
        custom_stopwords = [x.lower().strip('\n') for x in f.readlines()]
        custom_stopwords += list(ENGLISH_STOP_WORDS)

    # Créer un pipeline d'expérimentation pour le classifieur Naïve Bayes
    for n_features in n_features_values:
            pipeline_nb = Pipeline([
                ('tfidf', TfidfVectorizer(stop_words=custom_stopwords, ngram_range=ngram_values, max_features=n_features)),
                ('nb', MultinomialNB())
            ])

            # Validation croisée 
            nb_scores = cross_val_score(pipeline_nb, X, y, cv=stratified_kfold, scoring='accuracy')
            nb_predictions = cross_val_predict(pipeline_nb, X, y, cv=stratified_kfold)

            # Performances (rappel, précision, score F)
            results = {
                '% Incels' : int(ratio),
                'Algorithme' : 'Naive Bayes',
                'N traits discr.' : n_features,
                'accuracy': nb_scores.mean()       
            }
            results.update(classification_report(y, nb_predictions, output_dict=True)['macro avg'])
            report.append(results)
            print(classification_report(y, nb_predictions)) 
        


    # Tester différents paramètres pour le classifieur kNN
    for k in k_values:
        for n_features in n_features_values:
            # Créer un pipeline avec un classifieur kNN
            pipeline_knn = Pipeline([
                ('tfidf', TfidfVectorizer(stop_words=custom_stopwords, ngram_range=ngram_values, max_features=n_features)),
                ('knn', KNeighborsClassifier(n_neighbors=k))
            ])

            # Utiliser la validation croisée avec le classifieur kNN
            knn_scores = cross_val_score(pipeline_knn, X, y, cv=stratified_kfold, scoring='accuracy')
            knn_predictions = cross_val_predict(pipeline_knn, X, y, cv=stratified_kfold)

            # Performances moyennes sur les plis pour kNN
            results = {
                '% Incels' : int(ratio),
                'Algorithme' : f'knn (k={k})',
                'N traits discr.' : n_features,
                'accuracy': knn_scores.mean()    
            }

            results.update(classification_report(y, knn_predictions, output_dict=True)['macro avg'])
            report.append(results)
            print(classification_report(y, knn_predictions))

              precision    recall  f1-score   support

       incel       0.59      0.01      0.03      5000
     neutral       0.90      1.00      0.95     45000

    accuracy                           0.90     50000
   macro avg       0.75      0.51      0.49     50000
weighted avg       0.87      0.90      0.86     50000

              precision    recall  f1-score   support

       incel       0.61      0.02      0.04      5000
     neutral       0.90      1.00      0.95     45000

    accuracy                           0.90     50000
   macro avg       0.75      0.51      0.49     50000
weighted avg       0.87      0.90      0.86     50000

              precision    recall  f1-score   support

       incel       0.77      0.06      0.10      5000
     neutral       0.90      1.00      0.95     45000

    accuracy                           0.90     50000
   macro avg       0.84      0.53      0.53     50000
weighted avg       0.89      0.90      0.86     50000

              preci

c:\Users\camil\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\camil\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\camil\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

              precision    recall  f1-score   support

       incel       0.90      1.00      0.95     45000
     neutral       0.00      0.00      0.00      4999

    accuracy                           0.90     49999
   macro avg       0.45      0.50      0.47     49999
weighted avg       0.81      0.90      0.85     49999



c:\Users\camil\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\camil\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\camil\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

              precision    recall  f1-score   support

       incel       0.90      1.00      0.95     45000
     neutral       0.00      0.00      0.00      4999

    accuracy                           0.90     49999
   macro avg       0.45      0.50      0.47     49999
weighted avg       0.81      0.90      0.85     49999

              precision    recall  f1-score   support

       incel       0.90      1.00      0.95     45000
     neutral       0.33      0.00      0.00      4999

    accuracy                           0.90     49999
   macro avg       0.62      0.50      0.47     49999
weighted avg       0.84      0.90      0.85     49999

              precision    recall  f1-score   support

       incel       0.90      1.00      0.95     45000
     neutral       0.70      0.01      0.01      4999

    accuracy                           0.90     49999
   macro avg       0.80      0.50      0.48     49999
weighted avg       0.88      0.90      0.85     49999

              preci

**Résultats**

In [4]:
# N.B. : Support = Nombre d'occurrence dans chaque classe
report = pd.DataFrame(report)
report['nb_posts_incels'] = (report['% Incels'].apply(lambda x: x/100) * report['support']).astype(int)
report['nb_posts_neutral'] = (report['% Incels'].apply(lambda x: 1-(x/100)) * report['support']).astype(int)

report = report.rename(columns={'support':'nb_posts_total'})
report['nb_posts_total'] = report['nb_posts_total'].astype(int)

# Réorganiser l'ordre des colonnes
report = report[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme', 
                 'N traits discr.', 'accuracy', 'precision', 'recall','f1-score']]

report

,nb_posts_total,nb_posts_incels,nb_posts_neutral,% Incels,Algorithme,N traits discr.,accuracy,precision,recall,f1-score
0,50000.0,5000,45000,10,Naive Bayes,100,0.900420,0.745446,0.506367,0.487247
1,50000.0,5000,45000,10,Naive Bayes,250,0.900720,0.753804,0.509556,0.493749
2,50000.0,5000,45000,10,Naive Bayes,500,0.903920,0.838549,0.526889,0.526485
3,50000.0,5000,45000,10,Naive Bayes,1000,0.906600,0.863636,0.541267,0.551973
4,50000.0,5000,45000,10,Naive Bayes,1500,0.907380,0.859318,0.546944,0.561621
...,...,...,...,...,...,...,...,...,...,...
589,49999.0,44999,4999,90,knn (k=5),3000,0.870398,0.531627,0.514663,0.513839
590,49999.0,44999,4999,90,knn (k=5),4000,0.871838,0.538518,0.517685,0.517878
591,49999.0,44999,4999,90,knn (k=5),5000,0.871338,0.533608,0.515274,0.514557
592,49999.0,44999,4999,90,knn (k=5),10000,0.871138,0.529680,0.513295,0.511780


**Exporter les résultats de l'apprentissage**

In [9]:
report.to_csv(f'../3-results/results_training.csv', index=False)

**Tester le système retenu sur de nouvelles données**

*On reproduit l'apprentissage pour le système retenu uniquement*

In [ ]:
file_train = '../1-data/training_datasets/train_dataset_40pc.xlsx'

df_train = pd.read_excel(file_train)
df_train

In [ ]:
# Système retenu : Naive Bayes ; 40% de données Incels ; 5 000 traits discriminants
X_train = df_train['text_post'].astype(str)
y_train = df_train['category']

ngram_values = (1, 2)

# Antidictionnaire personnalisé
with open('./utils/exclusion_v2.stop', encoding='utf-8') as f:
    custom_stopwords = [x.lower().strip('\n') for x in f.readlines()]
    custom_stopwords += list(ENGLISH_STOP_WORDS)

pipeline_nb = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=custom_stopwords, ngram_range=ngram_values, max_features=15000)),
    ('nb', MultinomialNB())
])

pipeline_nb.fit(X_train, y_train)

In [ ]:
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
nb_predictions = cross_val_predict(pipeline_nb, X_train, y_train, cv=stratified_kfold)

# Performances en phase d'apprentissage du modèle retenu (rappel, précision, score F)
# Calculer et afficher les performances
accuracy_nb = accuracy_score(y_train, nb_predictions)
precision_nb = precision_score(y_train, nb_predictions, average='macro')
recall_nb = recall_score(y_train, nb_predictions, average='macro')
f1_nb = f1_score(y_train, nb_predictions, average='macro')

df = [
    {
        'Phase': 'Apprentissage',
        'Précision': precision_nb,
        'Rappel': recall_nb,
        'Mesure F': f1_nb,
        'Exactitude': accuracy_nb

    }
]

print(classification_report(y_train, nb_predictions))

*On teste sur de nouvelles données*

In [ ]:
file_test = '../1-data/test_dataset_10pc.xlsx'

df_test = pd.read_excel(file_test)
df_test

*Vérifier qu'aucun donnée test n'était contenue dans les données d'appentissage*

In [ ]:
validate = pd.concat([df_train, df_test])

validate[validate.duplicated(subset='id_post')]

*Appliquer le classifieur aux nouvelles données*

In [ ]:
X_test = df_test['text_post'].astype(str)
y_test = df_test['category']

y_pred_nb = pipeline_nb.predict(X_test)

In [ ]:
# Calculer et afficher les performances
accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb, average='macro')
recall_nb = recall_score(y_test, y_pred_nb, average='macro')
f1_nb = f1_score(y_test, y_pred_nb, average='macro')

df.append(
    {
        'Phase': 'Test',
        'Précision': precision_nb,
        'Rappel': recall_nb,
        'Mesure F': f1_nb,
        'Exactitude': accuracy_nb

    }
)

pd.set_option("display.precision", 4)
df = pd.DataFrame(df)
df.to_csv('../3-results/results_test.csv', index=False)
df

In [ ]:
accuracy_nb

In [ ]:
precision_nb

In [ ]:
recall_nb

In [ ]:
f1_nb

In [ ]:
print(classification_report(y_test, y_pred_nb))

In [ ]:
import pandas as pd
results = pd.read_csv('../3-results/results_training.csv')
results.sort_values(by='f_score', ascending=False)